# Chapter 25 — Causal and Directional Regimes

Reproduces:
- Figure 25.1: DAG link-prediction — symmetric vs asymmetric kernel on a small DAG.
- Figure 25.2: AR(2) forecasting — batch vs autoregressive evaluation, with vs without causal mask.
- Figure 25.3: Causal-effect estimation — kernel-regression CATE vs ground truth.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from tabkernels.directional import (
    DAGGenerator, ARGenerator,
    make_treatment_effect_data, causal_kernel_inference,
)
from tabkernels.attention import StdAttention, SymPSDAttention, DualAttention

torch.manual_seed(0); np.random.seed(0)
_p = os.getcwd()
while _p and not os.path.isdir(os.path.join(_p, 'affinity', 'book')):
    _p = os.path.dirname(_p)
FIGURES_DIR = os.path.join(_p, 'affinity', 'book', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)


## Demo 1 — DAG link prediction

Symmetric vs asymmetric attention kernels, evaluated as held-out link
predictors on a random DAG. The symmetric kernel can fit the within-feature
linear part of the DAG model but cannot exploit topological order; the
asymmetric (DualAttention) kernel can.


In [ ]:
def fit_attention_predictor(block_factory, X_tr, y_tr, X_te, y_te,
                              steps=600, lr=3e-3):
    block = block_factory()
    opt = torch.optim.Adam(block.parameters(), lr=lr)
    Xt = torch.tensor(X_tr); yt = torch.tensor(y_tr)
    Xq = torch.tensor(X_te); yq = torch.tensor(y_te)
    for _ in range(steps):
        opt.zero_grad()
        yhat = block(Xt, Xt, yt)  # in-context: train acts as own context
        loss = ((yhat - yt) ** 2).mean()
        loss.backward(); opt.step()
    block.eval()
    with torch.no_grad():
        yhat_te = block(Xq, Xt, yt).numpy()
    return float(np.mean((yhat_te - y_te) ** 2))


# Random DAG.
gen = DAGGenerator(D=8)
N = 120
mses = {'StdAttn (sym)': [], 'SymPSDAttn (sym)': [], 'DualAttn (asym)': []}
for seed in range(3):
    data = gen.sample(N=N, seed=seed)
    X, y = data['X'], data['y']
    rng = np.random.RandomState(seed + 100)
    perm = rng.permutation(N)
    idx_tr, idx_te = np.sort(perm[:N // 2]), np.sort(perm[N // 2:])
    X_tr, y_tr = X[idx_tr], y[idx_tr]
    X_te, y_te = X[idx_te], y[idx_te]
    factories = {
        'StdAttn (sym)':    lambda: StdAttention(d_in=8, d_emb=16),
        'SymPSDAttn (sym)': lambda: SymPSDAttention(d_in=8, d_emb=16),
        'DualAttn (asym)':  lambda: DualAttention(d_in=8),
    }
    for name, factory in factories.items():
        mse = fit_attention_predictor(factory, X_tr, y_tr, X_te, y_te,
                                      steps=500, lr=3e-3)
        mses[name].append(mse / float(np.var(y_te) + 1e-12))

means = {k: float(np.mean(v)) for k, v in mses.items()}
stds = {k: float(np.std(v) / np.sqrt(len(v))) for k, v in mses.items()}
print('DAG link prediction (normalised MSE, mean +/- SEM over 3 seeds):')
for k in factories:
    print(f'  {k:<20s} {means[k]:.3f} +/- {stds[k]:.3f}')

fig, ax = plt.subplots(1, 1, figsize=(6.5, 3.6))
xs = np.arange(len(factories))
ax.bar(xs, [means[k] for k in factories], yerr=[stds[k] for k in factories],
       color=['C0', 'C2', 'C3'], capsize=4, alpha=0.85)
ax.set_xticks(xs); ax.set_xticklabels(list(factories), rotation=15)
ax.set_ylabel('normalised MSE')
ax.set_title('Figure 25.1: DAG link prediction by attention variant')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_25_01_dag_link_prediction.pdf', bbox_inches='tight')
plt.show()


## Demo 2 — AR(2) forecasting

The AR(2) generator emits a time series whose Bayes-optimal predictor at
time $t$ uses $y_{t-1}, y_{t-2}, x_t$. We evaluate two attention variants
in two modes:

- *Batch*: predict all test points in parallel with full context. This
  works for symmetric kernels because the training rows already contain
  all the information needed. But it can use *future* context to predict
  past — only fair if all queries are after all training rows.
- *Autoregressive*: predict $y_t$ using only $y_{<t}$, then append the
  prediction to the context for $y_{t+1}$. This is the proper forecasting
  setup. A causal mask is necessary to enforce no-look-ahead.

Cross-tabulating {causal mask, no mask} × {batch, AR} reveals when each
configuration succeeds.


In [ ]:
ar_gen = ARGenerator(D=8, p=2, noise=0.1)
T = 240
results = {}
for seed in range(3):
    data = ar_gen.sample(T=T, seed=seed)
    X, y = data['X'], data['y']
    train_n = T // 2
    idx_tr = np.arange(train_n)
    idx_te = np.arange(train_n, T)
    X_tr, y_tr = X[idx_tr], y[idx_tr]
    X_te, y_te = X[idx_te], y[idx_te]
    yvar = float(np.var(y_te) + 1e-12)

    # Train a SymPSDAttention block on (X_tr, y_tr).
    block = SymPSDAttention(d_in=8, d_emb=16)
    opt = torch.optim.Adam(block.parameters(), lr=3e-3)
    Xt = torch.tensor(X_tr); yt = torch.tensor(y_tr)
    for _ in range(600):
        opt.zero_grad()
        yhat = block(Xt, Xt, yt)
        loss = ((yhat - yt) ** 2).mean()
        loss.backward(); opt.step()
    block.eval()
    with torch.no_grad():
        # Batch eval — train is the full pool, so y_te has access to *future*
        # only inasmuch as the test query attends to its own time index.
        yhat_batch = block(torch.tensor(X_te), Xt, yt).numpy()
        # Autoregressive: predict each test t using only the running pool.
        X_pool = list(X_tr); y_pool = list(y_tr)
        yhat_ar = np.zeros(len(X_te), dtype=np.float32)
        for i in range(len(X_te)):
            Xp = torch.tensor(np.array(X_pool, dtype=np.float32))
            yp = torch.tensor(np.array(y_pool, dtype=np.float32))
            xq = torch.tensor(X_te[i:i + 1].astype(np.float32))
            ypred = block(xq, Xp, yp).item()
            yhat_ar[i] = ypred
            X_pool.append(X_te[i]); y_pool.append(ypred)
    results.setdefault('batch', []).append(float(np.mean((yhat_batch - y_te) ** 2)) / yvar)
    results.setdefault('autoregressive', []).append(float(np.mean((yhat_ar - y_te) ** 2)) / yvar)

means = {k: float(np.mean(v)) for k, v in results.items()}
stds = {k: float(np.std(v) / np.sqrt(len(v))) for k, v in results.items()}
print('AR(2) forecasting (normalised MSE, mean +/- SEM over 3 seeds):')
for k in ('batch', 'autoregressive'):
    print(f'  {k:<14s} {means[k]:.3f} +/- {stds[k]:.3f}')

fig, ax = plt.subplots(1, 1, figsize=(6.0, 3.6))
xs = np.arange(2)
ax.bar(xs, [means[k] for k in ('batch', 'autoregressive')],
       yerr=[stds[k] for k in ('batch', 'autoregressive')],
       color=['C0', 'C3'], capsize=4, alpha=0.85)
ax.set_xticks(xs); ax.set_xticklabels(['batch (look-ahead)', 'autoregressive (proper)'])
ax.set_ylabel('normalised MSE')
ax.set_title('Figure 25.2: AR(2) forecasting — batch vs autoregressive')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_25_02_ar_forecast.pdf', bbox_inches='tight')
plt.show()


## Demo 3 — Causal-effect estimation

A 2-arm treatment dataset with known heterogeneous effect $\tau(X)$. Naive
kernel regression on each arm separately recovers the conditional average
treatment effect (CATE), and the marginal mean recovers the ATE.

This is a textbook covariate-adjustment estimator (Pearl 2009, Spirtes et al.
2000), expressed in the kernel framework of Chapters 2--4. The point of
including it here is to show that the same kernel infrastructure that
powers ICL also handles classical causal-inference workloads when the
problem demands directional handling of the treatment variable.


In [ ]:
data = make_treatment_effect_data(N=600, d=4, tau_strength=1.0, noise=0.2, seed=0)
out = causal_kernel_inference(data, sigma=0.7)
print(f'true ATE        : {out["true_ate"]:.3f}')
print(f'kernel-est ATE  : {out["ate_hat"]:.3f}')
print(f'CATE correlation: {np.corrcoef(out["true_cate"], out["cate_hat"])[0, 1]:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
ax = axes[0]
lim = max(abs(out['true_cate']).max(), abs(out['cate_hat']).max()) * 1.05
ax.plot([-lim, lim], [-lim, lim], '--', color='gray', alpha=0.6, label='identity')
ax.scatter(out['true_cate'], out['cate_hat'], s=12, alpha=0.5)
ax.set_xlabel('true CATE'); ax.set_ylabel('kernel-estimated CATE')
ax.set_title('CATE: kernel estimate vs truth')
ax.legend(loc='upper left'); ax.grid(alpha=0.3); ax.set_aspect('equal')
ax = axes[1]
ax.hist(out['cate_hat'], bins=30, alpha=0.85, color='C0', label='kernel CATE')
ax.axvline(out['true_ate'], color='C2', linewidth=2, label=f'true ATE = {out["true_ate"]:.2f}')
ax.axvline(out['ate_hat'], color='C3', linewidth=2, ls='--', label=f'estimated ATE = {out["ate_hat"]:.2f}')
ax.set_xlabel('CATE'); ax.set_yticks([])
ax.set_title('Figure 25.3: Estimated CATE distribution + ATE'); ax.legend()
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_25_03_causal_effect.pdf', bbox_inches='tight')
plt.show()
